# Project 2: Transformers

This project is part of the NLP module held in the spring of 2026. Three transformer models are compared, answering physical common sense tasks with the PIQA dataset. 

- **Randomly Initialized Transformer** 
- **Pretrained Transformer** not pretrained or finetuned on PIQA dataset
- **LLM (1B+ Parameters)** same hyperparameters

**Dataset**  
"PIQA: Reasoning about Physical Commonsense in Natural Language" — a binary choice task
where a model selects the more physically plausible solution to a given goal.  
Source: [https://arxiv.org/abs/1911.11641](https://arxiv.org/abs/1911.11641)

**Tools** 
- Course Materials
- Documentations (mostly of imported dependencies)
    - apxml
    - NLTK
    - PyTorch
    - skikit-learn
- Claude AI for the following tasks:
    - Helping formulate and clarify reasoning
    - General coding assistance
- Regex101
- Huggingface

**Weights & Biases**  
All experimental runs are logged and published in the
# TODO report

**Notebook structure**
1. Introduction
2. Setup
3. Preprocessing
4. Model
5. Training
6. Evaluation
7. Interpretation


## Setup

In [1]:
SKIP_TRAINING = True
SKIP_LLM_INFERENCE = True

In [2]:
!pip install \
    datasets==4.8.4 \
    numpy==2.4.4 \
    datetime==6.0.0 \
    transformers==5.6.2 \
    torch==2.10.0 \
    scikit-learn==1.8.0 \
    wandb==0.25.1


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
from datasets import load_dataset
from datetime import datetime
from transformers import AutoTokenizer, BertConfig, BertModel, AutoModelForCausalLM
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
import torch.nn as nn
import numpy as np
import wandb
import re
import torch

In [4]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
generator = torch.Generator()
generator.manual_seed(SEED)

In [5]:
TS = datetime.now().strftime("%Y%m%d_%H%M%S")

In [6]:
wandb_project = "nlp-project2-piqa"

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## Preprocessing

In [8]:
train_split = load_dataset("ybisk/piqa", split="train[:-1000]", revision='refs/convert/parquet')
valid_split = load_dataset("ybisk/piqa", split="train[-1000:]", revision='refs/convert/parquet')
test_split = load_dataset("ybisk/piqa", split="validation", revision='refs/convert/parquet')

### Feature Selection

In [9]:
COL_GOAL = 'goal'
COL_SOL1 = 'sol1'
COL_SOL2 = 'sol2'
COL_LABEL = 'label'

### Filter HTML Elements

In [10]:
# regex source: https://apxml.com/courses/nlp-fundamentals/chapter-1-nlp-text-processing-techniques/handling-text-noise
# verified with: https://regex101.com
regex_pattern = re.compile(r'<[^>]+>', re.IGNORECASE)
html_elements = 0

for split in [train_split, valid_split, test_split]:
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_GOAL])))
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_SOL1])))
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_SOL2])))

print(f"Number of HTML elements found: {html_elements}")

Number of HTML elements found: 0


### Input Format

In [11]:
MAX_INPUT_LENGTH = 553 # value found by analysis further down

COL_INPUT1 = 'input1'
COL_INPUT2 = 'input2'

ATTENTION_MASK = 'attention_mask'
INPUT_IDS = 'input_ids'
TOKEN_TYPE_IDS = 'token_type_ids'

COL_INPUT1_ATTENTION_MASK = f"{COL_INPUT1}_{ATTENTION_MASK}"
COL_INPUT1_INPUT_IDS = f"{COL_INPUT1}_{INPUT_IDS}"
COL_INPUT1_TOKEN_TYPE_IDS = f"{COL_INPUT1}_{TOKEN_TYPE_IDS}"
COL_INPUT2_ATTENTION_MASK = f"{COL_INPUT2}_{ATTENTION_MASK}"
COL_INPUT2_INPUT_IDS = f"{COL_INPUT2}_{INPUT_IDS}"
COL_INPUT2_TOKEN_TYPE_IDS  = f"{COL_INPUT2}_{TOKEN_TYPE_IDS}"

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def preprocess_row(row):
    tokenized1 = tokenizer(row[COL_GOAL], row[COL_SOL1], truncation=True, max_length=MAX_INPUT_LENGTH)
    tokenized2 = tokenizer(row[COL_GOAL], row[COL_SOL2], truncation=True, max_length=MAX_INPUT_LENGTH)
    return {
        COL_LABEL: row[COL_LABEL],
        COL_INPUT1_ATTENTION_MASK: tokenized1[ATTENTION_MASK],
        COL_INPUT1_INPUT_IDS: tokenized1[INPUT_IDS],
        COL_INPUT1_TOKEN_TYPE_IDS: tokenized1[TOKEN_TYPE_IDS],
        COL_INPUT2_ATTENTION_MASK: tokenized2[ATTENTION_MASK],
        COL_INPUT2_INPUT_IDS: tokenized2[INPUT_IDS],
        COL_INPUT2_TOKEN_TYPE_IDS: tokenized2[TOKEN_TYPE_IDS],
    }

train_processed = train_split.map(preprocess_row, remove_columns=train_split.column_names, batched=False)
valid_processed = valid_split.map(preprocess_row, remove_columns=valid_split.column_names, batched=False)
test_processed = test_split.map(preprocess_row,  remove_columns=test_split.column_names, batched=False)

print(train_processed[0])

{'label': 1, 'input1_attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'input1_input_ids': [101, 2043, 16018, 12136, 1010, 2043, 2009, 1005, 1055, 3201, 1010, 2017, 2064, 102, 10364, 2009, 3031, 1037, 5127, 102], 'input1_token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1], 'input2_attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'input2_input_ids': [101, 2043, 16018, 12136, 1010, 2043, 2009, 1005, 1055, 3201, 1010, 2017, 2064, 102, 10364, 2009, 2046, 1037, 15723, 102], 'input2_token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]}


### Length Analysis

In [12]:
all_input_lengths = (
    [len(row[COL_INPUT1_ATTENTION_MASK]) for row in train_processed] +
    [len(row[COL_INPUT2_ATTENTION_MASK]) for row in train_processed] +
    [len(row[COL_INPUT1_ATTENTION_MASK]) for row in valid_processed] +
    [len(row[COL_INPUT2_ATTENTION_MASK]) for row in valid_processed] +
    [len(row[COL_INPUT1_ATTENTION_MASK]) for row in test_processed] +
    [len(row[COL_INPUT2_ATTENTION_MASK]) for row in test_processed]
)

print(f"Max length of an input field: {np.max(all_input_lengths)}") 

Max length of an input field: 553


### Data Loader

In [13]:
BATCH_SIZE = 32

class PiqaDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        return {
            COL_INPUT1_INPUT_IDS: torch.tensor(row[COL_INPUT1_INPUT_IDS]),
            COL_INPUT1_ATTENTION_MASK: torch.tensor(row[COL_INPUT1_ATTENTION_MASK]),
            COL_INPUT1_TOKEN_TYPE_IDS: torch.tensor(row[COL_INPUT1_TOKEN_TYPE_IDS]),
            COL_INPUT2_INPUT_IDS: torch.tensor(row[COL_INPUT2_INPUT_IDS]),
            COL_INPUT2_ATTENTION_MASK: torch.tensor(row[COL_INPUT2_ATTENTION_MASK]),
            COL_INPUT2_TOKEN_TYPE_IDS: torch.tensor(row[COL_INPUT2_TOKEN_TYPE_IDS]),
            COL_LABEL: torch.tensor(row[COL_LABEL]),
        }
    
def pad_sequence(sequences):
    max_length_in_batch = max(len(s) for s in sequences)
    return torch.stack([
        torch.nn.functional.pad(s, (0, max_length_in_batch - len(s))) for s in sequences
    ])

def collate_fn(batch):
    input1_ids = pad_sequence([item[COL_INPUT1_INPUT_IDS] for item in batch])
    input1_attention_mask = pad_sequence([item[COL_INPUT1_ATTENTION_MASK] for item in batch])
    input1_token_type_ids = pad_sequence([item[COL_INPUT1_TOKEN_TYPE_IDS] for item in batch])
    input2_ids = pad_sequence([item[COL_INPUT2_INPUT_IDS] for item in batch])
    input2_attention_mask = pad_sequence([item[COL_INPUT2_ATTENTION_MASK] for item in batch])
    input2_token_type_ids = pad_sequence([item[COL_INPUT2_TOKEN_TYPE_IDS] for item in batch])
    labels = torch.tensor([item[COL_LABEL] for item in batch], dtype=torch.long)

    return (
        input1_ids, 
        input1_attention_mask, 
        input1_token_type_ids,
        input2_ids, 
        input2_attention_mask, 
        input2_token_type_ids,
        labels
    )

# The training set has to be shuffled to ensure random order in training which makes training more stable.  
train_loader = DataLoader(PiqaDataset(train_processed), batch_size=BATCH_SIZE, collate_fn=collate_fn, shuffle=True, generator=generator)
# Test and Validation should not be shuffled to ensure reproducibility and consistency of the model.
valid_loader = DataLoader(PiqaDataset(valid_processed), batch_size=BATCH_SIZE, collate_fn=collate_fn)
test_loader = DataLoader(PiqaDataset(test_processed), batch_size=BATCH_SIZE, collate_fn=collate_fn)

## Model

### Bert Classifier
to ensure same base architecture

In [14]:
class BertTransformerClassifier(nn.Module):

    def __init__(self, bert: BertModel, dropout_p: float):
        super().__init__()
        self.bert = bert
        hidden_size = bert.config.hidden_size
        input_hidden_dim = 2 * hidden_size
        classifier_hidden_dim = input_hidden_dim // 2

        self.classifier = nn.Sequential(
            nn.Linear(input_hidden_dim, classifier_hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(classifier_hidden_dim, 2),
        )

    def encode(self, input_ids, attention_mask, token_type_ids):
        output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        # output: (B, L, 768)
        # B: for the whole batch size
        # L: length of input (nr of tokens)
        # 768 dim vector which is a summary of the whole input but biased by the current token -> [CLS] does not influence the summary -> all tokens are weighed based on the importance of the information they carry -> we can use this dim only
        cls_vector = output.last_hidden_state[:, 0, :]
        return cls_vector
    
    def forward(self, 
                input1_ids, input1_attention_mask, input1_token_type_ids, 
                input2_ids, input2_attention_mask, input2_token_type_ids):
        cls1 = self.encode(input1_ids, input1_attention_mask, input1_token_type_ids)
        cls2 = self.encode(input2_ids, input2_attention_mask, input2_token_type_ids)
        # cls dim: 768
        # combined dim: 2 * 768
        combined = torch.cat([cls1, cls2], dim=-1)
        logits = self.classifier(combined)
        return logits

### Randomly Initialized Transformer

In [15]:
def get_randomly_initialized_transformer():
    # use default config to match pretrained transformer
    # defaults described here: https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertConfig
    bert_config_random = BertConfig()
    return BertModel(bert_config_random).to(device)

### Pretrained Transformer

In [16]:
def get_pretrained_transformer(): 
    return BertModel.from_pretrained("bert-base-uncased").to(device)

### LLM (1B+ Parameters)

In [17]:
def get_llm_with_tokenizer():
    llm_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
    llm_model = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=torch.float16).to(device)
    return llm_model, llm_tokenizer

## Training

In [18]:
MODEL1_NAME = f"random_{TS}"
MODEL2_NAME = f"pretrained_{TS}"
BEST_MODEL1_PATH = f"models/best_overall_{MODEL1_NAME}.pt"
BEST_MODEL2_PATH = f"models/best_overall_{MODEL2_NAME}.pt"
SWEEP_COUNT = 1

In [19]:
best_valid_acc_model1 = 0
best_valid_acc_model2 = 0

training_config = {
    'max_epochs': 1,
    'patience': 5,
}

# use same model architecture in model 1 and 2
# todo config
def create_sweep_config(model_name):
    return {
        "method": "bayes",
        "metric": {"name": f"{model_name}/valid_acc", "goal": "maximize"},
        "parameters": {
            "lr":                  {"values": [1e-3, 1e-4, 1e-5]},
            "weight_decay":        {"values": [1e-3, 1e-4, 1e-5]},
            "dropout_probability": {"values": [0.1, 0.3, 0.5]},
        },
    }

In [20]:
if not SKIP_TRAINING:
    wandb.login()
else: 
    print("training skipped")

training skipped


In [21]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, total_correct, total = 0, 0, 0

    for input1_ids, input1_mask, input1_tt, input2_ids, input2_mask, input2_tt, labels in loader:
        input1_ids, input1_mask, input1_tt = input1_ids.to(device), input1_mask.to(device), input1_tt.to(device)
        input2_ids, input2_mask, input2_tt = input2_ids.to(device), input2_mask.to(device), input2_tt.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        logits = model(input1_ids, input1_mask, input1_tt, input2_ids, input2_mask, input2_tt)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=-1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, total_correct / total

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total = 0, 0, 0

    with torch.no_grad():
        for input1_ids, input1_mask, input1_tt, input2_ids, input2_mask, input2_tt, labels in loader:
            input1_ids, input1_mask, input1_tt = input1_ids.to(device), input1_mask.to(device), input1_tt.to(device)
            input2_ids, input2_mask, input2_tt = input2_ids.to(device), input2_mask.to(device), input2_tt.to(device)
            labels = labels.to(device)
            
            logits = model(input1_ids, input1_mask, input1_tt, input2_ids, input2_mask, input2_tt)
            loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            total_correct += (logits.argmax(dim=-1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, total_correct / total

def save_checkpoint(model, config, valid_acc, path, label):
    torch.save({'model_state_dict': model.state_dict(), 'config': config}, path)
    print(f"New best {label} saved (valid_acc: {valid_acc:.4f})")

def train_model(model, config, model_name, wandb_run, train_loader, valid_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config['weight_decay']
    )
    best_valid_acc = 0
    patience_counter = 0
    
    for epoch in range(config['max_epochs']):
        print("____________________________________")
        print(f"Epoch {epoch + 1}/{config['max_epochs']}")
        
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
        valid_loss, valid_acc = eval_epoch(model, valid_loader, criterion)
        
        print(f"Train loss: {train_loss:.4f} - Train acc: {train_acc:.4f} | Valid loss: {valid_loss:.4f} - Valid acc: {valid_acc:.4f}")
        
        wandb_run.log({
            f"{model_name}/epoch": epoch + 1, 
            f"{model_name}/train_loss": train_loss,
            f"{model_name}/train_acc": train_acc,
            f"{model_name}/valid_loss": valid_loss,
            f"{model_name}/valid_acc": valid_acc,
        })
        
        if valid_acc > best_valid_acc:
            best_valid_acc = valid_acc
            patience_counter = 0
            save_checkpoint(model, config, valid_acc, config['model_path'], "run model")
        
            if model_name == MODEL1_NAME and valid_acc > best_valid_acc_model1:
                best_valid_acc_model1 = valid_acc
                save_checkpoint(model, config, valid_acc, BEST_MODEL1_PATH, f"overall {model_name}")
            elif model_name == MODEL2_NAME and valid_acc > best_valid_acc_model2:
                best_valid_acc_model2 = valid_acc
                save_checkpoint(model, config, valid_acc, BEST_MODEL2_PATH, f"overall {model_name}")
                
        else:
            patience_counter += 1
            print(f"No improvement ({patience_counter}/{config['patience']})")
        
        if patience_counter >= config['patience']:
            print("Early stopping triggered")
            break

In [22]:
def sweep_run_model1():
    with wandb.init(project=wandb_project, config=training_config, group=f"random_{TS}") as wandb_run:
        wandb_config = wandb_run.config
        
        run_name = f"{MODEL1_NAME}_lr{wandb.config.lr}_wd{wandb.config.weight_decay}_dp{wandb.config.dropout_probability}_{wandb_run.id[:4]}"
        
        config = {
            'lr': wandb_config.lr,
            'weight_decay': wandb_config.weight_decay,
            'max_epochs': training_config['max_epochs'],
            'patience': training_config['patience'],
            'model_path': f"models/{run_name}.pt"
        }
        
        model = BertTransformerClassifier(bert=get_randomly_initialized_transformer(), dropout_p=wandb_config.dropout_probability).to(device)

        train_model(model, config, MODEL1_NAME, wandb_run, train_loader, valid_loader)
        
        del model
        torch.cuda.empty_cache()
        
        
if not SKIP_TRAINING:
    sweep_model1 = wandb.sweep(sweep=create_sweep_config(MODEL1_NAME), project=wandb_project)
    wandb.agent(sweep_model1, function=sweep_run_model1, count=SWEEP_COUNT)
else: 
    print("training skipped")

training skipped


In [23]:
def sweep_run_model2():
    with wandb.init(project=wandb_project, config=training_config, group=f"pretrained_{TS}") as wandb_run:
        wandb_config = wandb_run.config
        
        run_name = f"{MODEL2_NAME}_lr{wandb.config.lr}_wd{wandb.config.weight_decay}_dp{wandb.config.dropout_probability}_{wandb_run.id[:4]}"
        
        config = {
            'lr': wandb_config.lr,
            'weight_decay': wandb_config.weight_decay,
            'max_epochs': training_config['max_epochs'],
            'patience': training_config['patience'],
            'model_path': f"models/{run_name}.pt"
        }
        
        model = BertTransformerClassifier(bert=get_pretrained_transformer(), dropout_p=wandb_config.dropout_probability).to(device)
        
        train_model(model, config, MODEL2_NAME, wandb_run, train_loader, valid_loader)    
        
        del model
        torch.cuda.empty_cache()
        

if not SKIP_TRAINING:
    sweep_model2 = wandb.sweep(sweep=create_sweep_config(MODEL2_NAME), project=wandb_project)
    wandb.agent(sweep_model2, function=sweep_run_model2, count=SWEEP_COUNT)
else: 
    print("training skipped")

training skipped


### LLM Prompting

In [24]:
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
PROMPT_TEMPLATE = """You are answering a physical commonsense question. Given a goal and two possible solutions, choose the more physically plausible one.

Goal: {goal}
Solution 1: {sol1}
Solution 2: {sol2}

Answer with only "1" or "2"."""

def run_llm_inference(dataset):
    llm_model, llm_tokenizer = get_llm_with_tokenizer()
    predictions = []
    unparseable = 0

    for row in dataset:
        prompt = PROMPT_TEMPLATE.format(goal=row[COL_GOAL], sol1=row[COL_SOL1], sol2=row[COL_SOL2])

        # build input ids from the message 
        # with attention mask
        inputs = llm_tokenizer(prompt, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            # the generated output as ids 
            # with the input 
            output_ids = llm_model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=5,
                do_sample=False, # greedy prediction, so that it always predicts the highest token to ensure reproducability
            )

        # take the first output ids as there was only one message sent 
        # and remove the input ids because they are represented here as well in the output
        generated = output_ids[0][inputs["input_ids"].shape[-1]:]
        
        # the ids get decoded into the predicted label
        decoded = llm_tokenizer.decode(generated, skip_special_tokens=True)
        
        stripped = decoded.strip()
        if stripped.startswith("1"):
            predicted_label = 0
        elif stripped.startswith("2"):
            predicted_label = 1
        else:
            predicted_label = -1
            unparseable += 1

        predictions.append(predicted_label)

    print(f"Unparseable outputs: {unparseable}/{len(dataset)}")
    return predictions

In [25]:
if not SKIP_LLM_INFERENCE:
    llm_predictions = run_llm_inference(test_split)
    print(f"predictions: {llm_predictions}")
else:
    print("llm inference skipped")

llm inference skipped


## Evaluation

In [26]:
BEST_MODEL1_PATH = "models/best_overall_random.pt"
BEST_MODEL2_PATH = "models/best_overall_pretrained.pt"

In [27]:
def load_best_model(path, bert):
    checkpoint = torch.load(path)
    model = BertTransformerClassifier(bert=bert, dropout_p=checkpoint['config']['dropout_probability'])
    model.load_state_dict(checkpoint['model_state_dict'])
    return model.to(device)

def evaluate_transformer(model, loader):
    model.eval()
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for input1_ids, input1_mask, input1_tt, input2_ids, input2_mask, input2_tt, labels in loader:
            input1_ids, input1_mask, input1_tt = input1_ids.to(device), input1_mask.to(device), input1_tt.to(device)
            input2_ids, input2_mask, input2_tt = input2_ids.to(device), input2_mask.to(device), input2_tt.to(device)
            labels = labels.to(device)
            
            logits = model(input1_ids, input1_mask, input1_tt, input2_ids, input2_mask, input2_tt)
            predictions = logits.argmax(dim=-1)
            all_predictions.extend(predictions.tolist())
            all_labels.extend(labels.tolist())

    print(classification_report(all_labels, all_predictions, target_names=["sol1 correct", "sol2 correct"]))
    return all_predictions, all_labels

### Random Initialized Transformer

In [28]:
if not SKIP_TRAINING:
    best_model1 = load_best_model(BEST_MODEL1_PATH, get_randomly_initialized_transformer())
    predictions1, labels1 = evaluate_transformer(best_model1, test_loader)
    del best_model1
else:
    print("training skipped")

training skipped


### Pretrained Transformer

In [29]:
if not SKIP_TRAINING:
    best_model2 = load_best_model(BEST_MODEL2_PATH, get_pretrained_transformer())
    predictions2, labels2 = evaluate_transformer(best_model2, test_loader)
    del best_model2
else:
    print("training skipped")

training skipped


### Transformer Results Log

In [30]:
if not SKIP_TRAINING:
    with wandb.init(project=wandb_project, name=f"eval_{TS}"):
        wandb.log({
            f"{MODEL1_NAME}/test_acc": accuracy_score(labels1, predictions1),
            f"{MODEL2_NAME}/test_acc": accuracy_score(labels2, predictions2),
        })
else:
    print("training skipped")

training skipped


### Large Language Model

In [31]:
if not SKIP_LLM_INFERENCE:
    labels_test_split = [row[COL_LABEL] for row in test_split]
    print(classification_report(labels_test_split, llm_predictions, target_names=["sol1 correct", "sol2 correct"]))
else:
    print("llm inference skipped")

llm inference skipped


## Interpretation